# Video-Agent — HF Inference demo (optional, KHÔNG train)

Notebook optional: gọi **Hugging Face Serverless Inference** (`Qwen/Qwen2.5-72B-Instruct`) để thử nghiệm scriptwriter — cùng model/default với `backend/app/providers/hf_llm.py`.

- Không train/finetune weight nào.
- Cần set **Colab Secrets**: `HF_TOKEN` (Runtime → Secrets).
- Cell 1 clone repo về `/content/Video-Agent`; mọi path đều là `/content/...`, không dùng path máy local.
- Chi tiết: xem `README.md` cùng thư mục.

In [ ]:
# Cell 1 — Clone repo Video-Agent (thay VIDEO_REPO_URL bằng remote thật khi đã push)
# NOTE: repo local chưa có git remote (`git remote -v` trống) nên dùng placeholder;
# thay bằng URL GitHub thật của Video-Agent khi có.
VIDEO_REPO_URL = 'https://github.com/<GITHUB_USER>/Video-Agent.git'  # TODO: thay bằng remote thật
REPO_DIR = '/content/Video-Agent'

import os
if not os.path.isdir(REPO_DIR):
    !git clone $VIDEO_REPO_URL $REPO_DIR
else:
    print('Repo đã tồn tại, pull mới nhất...')
    !git -C $REPO_DIR pull --ff-only || true

%cd /content/Video-Agent
!ls


In [ ]:
# !pip install -q huggingface_hub
import json, os, re
from google.colab import userdata
from huggingface_hub import InferenceClient

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # set trong Colab Secrets
client = InferenceClient(api_key=os.environ['HF_TOKEN'])
MODEL = 'Qwen/Qwen2.5-72B-Instruct'  # cùng default với backend/app/providers/hf_llm.py

topic = 'AI giúp freelancer tăng thu nhập'
res = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'Bạn là Creative Director chuyên video ngắn TikTok/Shorts. Chỉ trả về JSON thuần: list 3 objects {scene_id, title, voiceover, visual_prompt, duration_sec}, không giải thích.'},
        {'role': 'user', 'content': f'Chủ đề video: {topic}'},
    ],
    max_tokens=800,
    temperature=0.7,
)
raw = res.choices[0].message.content.strip()
m = re.search(r'\[\s*\{.*\}\s*\]', raw, re.DOTALL)
scenes = json.loads(m.group(0)) if m else json.loads(raw)
print(json.dumps(scenes, indent=2, ensure_ascii=False))